# Bike-share demand: exploratory analysis

Synthetic daily ride counts for five stations during 2025, with temperature and rain.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

In [2]:
rng = np.random.default_rng(42)
days = pd.date_range("2025-01-01", "2025-12-31", freq="D")
stations = {"Downtown": 420, "University": 310, "Harbor": 260, "Old Town": 200, "Airport": 150}

doy = days.day_of_year.to_numpy()
temp = 12 + 10 * np.sin(2 * np.pi * (doy - 110) / 365) + rng.normal(0, 2.5, len(days))
rain = rng.gamma(0.6, 3.0, len(days)) * (rng.random(len(days)) < 0.35)

frames = []
for name, base in stations.items():
    weekend = 1.25 if name == "Harbor" else 0.75
    level = base * np.where(days.day_of_week < 5, 1.0, weekend)
    expected = level * (1 + 0.025 * (temp - 12)) * np.exp(-0.08 * rain)
    frames.append(pd.DataFrame({
        "date": days, "station": name,
        "rides": rng.poisson(np.clip(expected, 5, None)),
        "temp_c": temp.round(1), "rain_mm": rain.round(1),
    }))
df = pd.concat(frames, ignore_index=True)
df.shape

(1825, 5)

In [3]:
df.head()

,date,station,rides,temp_c,rain_mm
0,2025-01-01,Downtown,354,3.2,0.0
1,2025-01-02,Downtown,156,-0.2,7.6
2,2025-01-03,Downtown,344,4.2,0.0
3,2025-01-04,Downtown,250,4.7,0.0
4,2025-01-05,Downtown,190,-2.6,0.1


## Daily rides per station

In [4]:
fig = px.line(df, x="date", y="rides", color="station", title="Daily rides per station")
fig.show()

In [5]:
daily = df.groupby("date", as_index=False).agg(
    rides=("rides", "sum"), temp_c=("temp_c", "first"), rain_mm=("rain_mm", "first"))
fig = px.scatter(daily, x="temp_c", y="rides", color="rain_mm", color_continuous_scale="Blues",
                 title="Total rides vs. temperature",
                 labels={"temp_c": "temperature (°C)", "rain_mm": "rain (mm)"})
fig.show()

In [6]:
hours = np.arange(24)
commute = np.exp(-((hours - 8) ** 2) / 3) + np.exp(-((hours - 17.5) ** 2) / 4)
leisure = np.exp(-((hours - 14) ** 2) / 12)
profile = np.array([0.8 * commute + 0.3 * leisure if d < 5 else leisure for d in range(7)])
fig = go.Figure(go.Heatmap(z=profile.round(3), x=hours, y=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
                           colorscale="Viridis"))
fig.update_layout(title="Relative demand by hour and weekday", xaxis_title="hour of day")
fig.show()

In [7]:
top = df.groupby("station")["ride"].sum().sort_values(ascending=False)
top

KeyError: 'Column not found: ride'

In [8]:
df.describe()

,date,rides,temp_c,rain_mm
count,1825,1825.000000,1825.000000,1825.000000
mean,2025-07-02 00:00:00,246.626301,11.977534,0.603562
min,2025-01-01 00:00:00,9.000000,-2.600000,0.000000
25%,2025-04-02 00:00:00,163.000000,5.200000,0.000000
50%,2025-07-02 00:00:00,233.000000,11.900000,0.000000
75%,2025-10-01 00:00:00,315.000000,18.200000,0.200000
max,2025-12-31 00:00:00,604.000000,27.300000,22.300000
std,NaN,106.964053,7.399835,1.909296


## Notes

- Weekday commuting dominates Downtown and University.
- Rain suppresses demand at every station.